# 14 — Live Simulation (Updated with Completed Matches)

Re-run this notebook as the 2026 World Cup progresses.
Edit `COMPLETED_RESULTS` with real scores, rerun all cells, get updated probabilities.

**This does NOT overwrite `simulation_results_v2.csv`** — saves to `simulation_results_live.csv` instead, so your original pre-tournament prediction stays intact.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from collections import defaultdict
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. Load data & models (v2)

In [ ]:
fixtures = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\fixtures_clean.csv")
groups   = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\group_stages_clean.csv")
elo      = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\elo_all_teams_v2.csv")
ranks    = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\fifa_rankings.csv")

with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\home_model_v2.pkl", 'rb') as f:
    home_model = pickle.load(f)
with open(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\away_model_v2.pkl", 'rb') as f:
    away_model = pickle.load(f)

team_info = elo.merge(ranks, left_on='team', right_on='Nation', how='left').drop(columns='Nation')
team_info = team_info.rename(columns={'FIFA_2026_rank': 'fifa_rank'})
team_dict = team_info.set_index('team')[['elo','fifa_rank']].to_dict(orient='index')
for t in team_dict:
    if pd.isna(team_dict[t]['fifa_rank']):
        team_dict[t]['fifa_rank'] = 100

print("Loaded!")

## 2. ✏️ EDIT THIS — Completed match results

Add one entry per completed match. Format:
```python
{'home_team': 'Spain', 'away_team': 'Morocco', 'home_score': 3, 'away_score': 0}
```
Order doesn't need to match the fixture's home/away — it's matched by team names either way.

In [ ]:
COMPLETED_RESULTS = [
    # Example — replace with real results:
    # {'home_team': 'Spain', 'away_team': 'Morocco', 'home_score': 3, 'away_score': 0},
    # {'home_team': 'Argentina', 'away_team': 'Uzbekistan', 'home_score': 2, 'away_score': 0},
]

print(f"{len(COMPLETED_RESULTS)} completed matches entered.")

## 3. Prediction & simulation functions (v2 + Dixon-Coles)

In [ ]:
RHO = -0.13

def dc_correction(i, j, hxg, axg, rho=RHO):
    if   i == 0 and j == 0: return 1 - hxg * axg * rho
    elif i == 1 and j == 0: return 1 + axg * rho
    elif i == 0 and j == 1: return 1 + hxg * rho
    elif i == 1 and j == 1: return 1 - rho
    return 1.0

def predict_match(home_team, away_team, neutral=True, tw=5.0):
    he, ae = team_dict[home_team]['elo'], team_dict[away_team]['elo']
    hr, ar = team_dict[home_team]['fifa_rank'], team_dict[away_team]['fifa_rank']
    row = pd.DataFrame([{'home_elo': he, 'away_elo': ae, 'elo_diff': he-ae,
                          'neutral': int(neutral), 'tournament_weight': tw,
                          'fifa_rank_diff': hr-ar}])
    return home_model.predict(row)[0], away_model.predict(row)[0]

def simulate_match(ht, at, neutral=True, tw=5.0):
    hxg, axg = predict_match(ht, at, neutral, tw)
    return np.random.poisson(hxg), np.random.poisson(axg)

def simulate_knockout(t1, t2):
    hxg, axg = predict_match(t1, t2, neutral=True)
    hg, ag = np.random.poisson(hxg), np.random.poisson(axg)
    if hg == ag:
        hg += np.random.poisson(hxg*0.33); ag += np.random.poisson(axg*0.33)
    if hg == ag:
        pen = np.clip(0.5 + (team_dict[t1]['elo']-team_dict[t2]['elo'])*0.0001, 0.4, 0.6)
        if np.random.random() < pen: hg += 1
        else: ag += 1
    return (t1 if hg > ag else t2), hg, ag

print("Functions ready!")

## 4. Match completed results to fixtures

In [ ]:
def build_completed_lookup(completed_results, fixtures_df):
    """Map each completed result to its fixture, return set of completed (sorted team pair) tuples
    and a dict of {frozenset({team1,team2}): (home_score, away_score, home_team, away_team)}"""
    lookup = {}
    matched, unmatched = [], []

    for res in completed_results:
        ht, at = res['home_team'], res['away_team']
        hs, as_ = res['home_score'], res['away_score']
        pair = frozenset({ht, at})

        fixture_match = fixtures_df[
            (fixtures_df['home_team'].isin([ht, at])) &
            (fixtures_df['away_team'].isin([ht, at])) &
            (fixtures_df['home_team'] != fixtures_df['away_team'])
        ]
        fixture_match = fixture_match[
            fixture_match.apply(lambda r: frozenset({r['home_team'], r['away_team']}) == pair, axis=1)
        ]

        if len(fixture_match) == 0:
            unmatched.append(res)
            continue

        lookup[pair] = (ht, at, hs, as_)
        matched.append(res)

    return lookup, matched, unmatched

completed_lookup, matched_results, unmatched_results = build_completed_lookup(COMPLETED_RESULTS, fixtures)

print(f"Matched:   {len(matched_results)}")
print(f"Unmatched: {len(unmatched_results)}")
if unmatched_results:
    print("\n⚠️  Could not match these — check team names:")
    for r in unmatched_results:
        print(f"  {r}")

## 5. Group simulation using completed results

In [ ]:
def simulate_group_live(group_name, completed_lookup):
    teams = groups[groups['group'] == group_name]['nation'].tolist()
    table = {t: {'W':0,'D':0,'L':0,'GF':0,'GA':0,'Pts':0} for t in teams}

    group_fixtures = fixtures[
        fixtures['home_team'].isin(teams) & fixtures['away_team'].isin(teams)
    ]

    for _, row in group_fixtures.iterrows():
        h, a = row['home_team'], row['away_team']
        pair = frozenset({h, a})

        if pair in completed_lookup:
            # Use real result
            ch, ca, chs, cas = completed_lookup[pair]
            # Map back onto this fixture's home/away orientation
            if ch == h:
                hg, ag = chs, cas
            else:
                hg, ag = cas, chs
        else:
            # Simulate
            hg, ag = simulate_match(h, a, neutral=row.get('neutral', True))

        table[h]['GF']+=hg; table[h]['GA']+=ag
        table[a]['GF']+=ag; table[a]['GA']+=hg
        if hg>ag: table[h]['W']+=1; table[h]['Pts']+=3; table[a]['L']+=1
        elif hg<ag: table[a]['W']+=1; table[a]['Pts']+=3; table[h]['L']+=1
        else: table[h]['D']+=1; table[a]['D']+=1; table[h]['Pts']+=1; table[a]['Pts']+=1

    tdf = pd.DataFrame(table).T
    tdf['GD'] = tdf['GF'] - tdf['GA']
    return tdf.sort_values(['Pts','GD','GF'], ascending=False).reset_index().rename(columns={'index':'Team'})

# Quick check — current state of each group
print("Current group states (completed results applied, rest simulated once):")
for g in sorted(groups['group'].unique()):
    t = simulate_group_live(g, completed_lookup)
    print(f"\nGroup {g}:")
    print(t[['Team','Pts','GD','GF','GA']].to_string(index=False))

## 6. Qualifiers

In [ ]:
def get_qualifiers_live(completed_lookup):
    winners, runners, thirds = [], [], []
    for g in sorted(groups['group'].unique()):
        t = simulate_group_live(g, completed_lookup)
        winners.append(t.iloc[0]['Team'])
        runners.append(t.iloc[1]['Team'])
        thirds.append({'team': t.iloc[2]['Team'], 'pts': t.iloc[2]['Pts'],
                       'gd': t.iloc[2]['GD'], 'gf': t.iloc[2]['GF']})
    best_thirds = pd.DataFrame(thirds).sort_values(['pts','gd','gf'], ascending=False).head(8)['team'].tolist()
    return winners + runners + best_thirds

print("get_qualifiers_live ready!")

## 7. Knockout simulation

In [ ]:
def simulate_knockout_rounds(teams):
    rounds = {'Round of 32': [], 'Round of 16': [], 'Quarter-finals': [],
              'Semi-finals': [], 'Final': [], 'Winner': None}
    current_round = teams.copy()
    round_names = ['Round of 32','Round of 16','Quarter-finals','Semi-finals','Final']

    for round_name in round_names:
        next_round = []
        np.random.shuffle(current_round)
        for i in range(0, len(current_round), 2):
            t1, t2 = current_round[i], current_round[i+1]
            winner, hg, ag = simulate_knockout(t1, t2)
            next_round.append(winner)
            rounds[round_name].append(winner)
        current_round = next_round

    rounds['Winner'] = current_round[0]
    return rounds

print("simulate_knockout_rounds ready!")

## 8. Monte Carlo — 10,000 simulations with live results baked in

In [ ]:
N_SIMULATIONS = 10000
stage_counts = defaultdict(lambda: defaultdict(int))

print(f"Running {N_SIMULATIONS:,} live simulations...")

for sim in range(N_SIMULATIONS):
    if sim % 1000 == 0:
        print(f"  {sim}/{N_SIMULATIONS}...")

    qualifiers = get_qualifiers_live(completed_lookup)
    for team in qualifiers:
        stage_counts[team]['Round of 32'] += 1

    result = simulate_knockout_rounds(qualifiers)
    for round_name in ['Round of 16','Quarter-finals','Semi-finals','Final']:
        for team in result[round_name]:
            stage_counts[team][round_name] += 1
    stage_counts[result['Winner']]['Winner'] += 1

print("Done!")

## 9. Results & comparison with pre-tournament odds

In [ ]:
all_teams = groups['nation'].tolist()
results = []
for team in all_teams:
    results.append({
        'Team': team,
        'R32 %':   round(stage_counts[team]['Round of 32']    / N_SIMULATIONS * 100, 1),
        'R16 %':   round(stage_counts[team]['Round of 16']    / N_SIMULATIONS * 100, 1),
        'QF %':    round(stage_counts[team]['Quarter-finals'] / N_SIMULATIONS * 100, 1),
        'SF %':    round(stage_counts[team]['Semi-finals']    / N_SIMULATIONS * 100, 1),
        'Final %': round(stage_counts[team]['Final']          / N_SIMULATIONS * 100, 1),
        'Winner %':round(stage_counts[team]['Winner']         / N_SIMULATIONS * 100, 1),
    })

live_df = pd.DataFrame(results).sort_values('Winner %', ascending=False).reset_index(drop=True)
live_df.index += 1

print("LIVE simulation results (with completed matches applied):")
print(live_df.head(20).to_string())

In [ ]:
# Compare with pre-tournament baseline
try:
    baseline = pd.read_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\simulation_results_v2.csv")
    comparison = live_df.merge(baseline[['Team','Winner %']], on='Team', suffixes=('_live','_pre'))
    comparison['change'] = (comparison['Winner %_live'] - comparison['Winner %_pre']).round(1)
    comparison = comparison.sort_values('Winner %_live', ascending=False).reset_index(drop=True)
    comparison.index += 1

    print("Winner % — Live vs Pre-tournament:")
    print(comparison[['Team','Winner %_pre','Winner %_live','change']].head(20).to_string())
except FileNotFoundError:
    print("simulation_results_v2.csv not found — skipping comparison")

## 10. Save

In [ ]:
live_df.to_csv(r"C:\Users\aryan\OneDrive\Desktop\World Cup Predictor\simulation_results_live.csv", index=False)
print("Saved simulation_results_live.csv")
print("(simulation_results_v2.csv untouched — your original prediction is safe)")